In [9]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import os
# THÊM: Import thư viện mã hóa và chuẩn hóa
from sklearn.preprocessing import LabelEncoder, StandardScaler 

# Đảm bảo thư mục data tồn tại
os.makedirs('../data', exist_ok=True)

def generate_synthetic_data(num_samples=500000):
    print(f"Đang tạo {num_samples} mẫu dữ liệu tổng hợp...")
    np.random.seed(42)
    
    # 1. Các thuộc tính giao thoa (Intersectional attributes)
    gender = np.random.choice(['Male', 'Female'], size=num_samples)
    race = np.random.choice(['White', 'Black', 'Asian', 'Hispanic'], size=num_samples, p=[0.5, 0.2, 0.15, 0.15])
    age = np.random.randint(18, 70, size=num_samples)
    
    # SỬA/THÊM: Thêm tác động của race vào education để đồ thị nhân quả thực tế hơn
    edu_penalty = np.where(np.isin(race, ['Black', 'Hispanic']), -1.5, 0)
    education_years = np.random.normal(12 + edu_penalty, 3, size=num_samples).clip(8, 20)
    
    # 2. Tạo Bias Pathway (Đường dẫn thiên kiến nhân quả)
    base_income = 30000 + (education_years * 3000) + (age * 200)
    gender_penalty = np.where(gender == 'Female', -5000, 0)
    race_penalty = np.where(np.isin(race, ['Black', 'Hispanic']), -4000, 0)
    
    income = base_income + gender_penalty + race_penalty + np.random.normal(0, 5000, size=num_samples)
    
    # 3. Biến mục tiêu: Phê duyệt tín dụng (Credit Approval)
    prob_approval = 1 / (1 + np.exp(-(income - 45000) / 10000))
    loan_approved = np.random.binomial(1, prob_approval)
    
    df = pd.DataFrame({
        'gender': gender,
        'race': race,
        'age': age,
        'education_years': education_years.astype(int),
        'income': income.round(2),
        'loan_approved': loan_approved
    })
    
    # THÊM: Mã hóa Label Encoding cho Text để NOTEARS không bị lỗi
    le = LabelEncoder()
    df['gender'] = le.fit_transform(df['gender'])
    df['race'] = le.fit_transform(df['race'])
    
    # THÊM: Chuẩn hóa Scaling cho các biến liên tục chống Gradient Explosion
    scaler = StandardScaler()
    df[['age', 'education_years', 'income']] = scaler.fit_transform(df[['age', 'education_years', 'income']])
    
    # Lưu ra file CSV
    file_path = '../data/synthetic_intersectional_data.csv'
    df.to_csv(file_path, index=False)
    print(f"✅ Đã lưu dữ liệu tổng hợp tại: {file_path}")
    
    return df.head()

# Thực thi hàm
df_synthetic = generate_synthetic_data()
df_synthetic

Đang tạo 500000 mẫu dữ liệu tổng hợp...
✅ Đã lưu dữ liệu tổng hợp tại: ../data/synthetic_intersectional_data.csv


,gender,race,age,education_years,income,loan_approved
0,1,2,1.699939,-0.841885,-0.745092,1
1,0,3,1.500075,-1.217864,0.230608,1
2,1,3,-0.431944,0.286051,0.289860,1
3,1,1,-0.698429,0.662030,-0.410285,1
4,1,3,0.101027,-0.841885,0.008675,1


In [10]:
import pandas as pd
import urllib.request
import os
# THÊM: Import thư viện mã hóa và chuẩn hóa
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- 1. Xử lý German Credit Data ---
print("Đang tải và xử lý German Credit Data...")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = ['status', 'duration', 'credit_history', 'purpose', 'credit_amount', 
           'savings', 'employment', 'installment_rate', 'personal_status_sex', 
           'other_debtors', 'residence_since', 'property', 'age', 'other_installment_plans', 
           'housing', 'existing_credits', 'job', 'num_dependents', 'telephone', 'foreign_worker', 'target']
df_german = pd.read_csv(url, sep=' ', header=None, names=columns)

# Chuyển đổi giới tính từ mã của UCI và target (1: Tốt, 0: Xấu)
df_german['gender'] = df_german['personal_status_sex'].apply(lambda x: 'Female' if x in ['A92', 'A95'] else 'Male')
df_german['target'] = df_german['target'].apply(lambda x: 1 if x == 1 else 0) 

# THÊM: Encode tất cả các cột object (chữ) còn lại thành số
le = LabelEncoder()
for col in df_german.select_dtypes(include=['object', 'str']).columns:
    df_german[col] = le.fit_transform(df_german[col].astype(str))

# THÊM: Chuẩn hóa các biến số
scaler = StandardScaler()
num_cols = ['duration', 'credit_amount', 'installment_rate', 'residence_since', 'age', 'existing_credits', 'num_dependents']
df_german[num_cols] = scaler.fit_transform(df_german[num_cols])

df_german.to_csv('../data/german_credit_processed.csv', index=False)
print("✅ Đã lưu German Credit Data.")
display(df_german.head(3))

# --- 2. Xử lý Home Credit Data ---
file_path = '../data/application_train.csv'
if os.path.exists(file_path):
    print("Đang xử lý Home Credit Data...")
    df_home = pd.read_csv(file_path)
    
    # Lấy các cột quan trọng cho fairness
    cols_to_keep = [
    'TARGET', 
    'CODE_GENDER', 
    'DAYS_BIRTH', # Sẽ đổi thành age
    'AMT_INCOME_TOTAL', 
    'NAME_EDUCATION_TYPE',
    'REGION_RATING_CLIENT', # Bổ sung vùng miền theo đúng đề cương
    'NAME_INCOME_TYPE',     # Loại thu nhập (Làm công, thương gia...)
    'NAME_FAMILY_STATUS',   # Tình trạng hôn nhân
    'OCCUPATION_TYPE',      # Nghề nghiệp
    'AMT_CREDIT'            # Số tiền muốn vay
]
    df_home = df_home[cols_to_keep].copy()
    
    # Đổi DAYS_BIRTH thành tuổi (age)
    df_home['age'] = (df_home['DAYS_BIRTH'] / -365).astype(int)
    df_home.drop(columns=['DAYS_BIRTH'], inplace=True)
    df_home = df_home[df_home['CODE_GENDER'] != 'XNA'] # Xóa giới tính không xác định
    
    # THÊM: Mã hóa Label Encoding
    df_home['CODE_GENDER'] = le.fit_transform(df_home['CODE_GENDER'])
    df_home['NAME_EDUCATION_TYPE'] = le.fit_transform(df_home['NAME_EDUCATION_TYPE'])
    
    # THÊM: Chuẩn hóa Scaling
    df_home[['AMT_INCOME_TOTAL', 'age']] = scaler.fit_transform(df_home[['AMT_INCOME_TOTAL', 'age']])
    
    df_home.to_csv('../data/home_credit_processed.csv', index=False)
    print("✅ Đã lưu Home Credit Data.")
    display(df_home.head(3))
else:
    print(f"❌ KHÔNG TÌM THẤY FILE! Vui lòng tải application_train.csv từ Kaggle và đặt vào thư mục: {file_path}")

Đang tải và xử lý German Credit Data...
✅ Đã lưu German Credit Data.


,status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,other_debtors,...,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target,gender
0,0,-1.236478,4,4,-0.745131,4,4,0.918477,2,0,...,2.766456,2,1,1.027079,2,-0.428290,1,0,1,1
1,1,2.248194,2,4,0.949817,0,2,-0.870183,1,0,...,-1.191404,2,1,-0.704926,2,-0.428290,0,0,0,0
2,3,-0.738668,4,7,-0.416562,0,3,-0.870183,2,0,...,1.183312,2,1,-0.704926,1,2.334869,0,0,1,1


Đang xử lý Home Credit Data...
✅ Đã lưu Home Credit Data.


,TARGET,CODE_GENDER,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,REGION_RATING_CLIENT,NAME_INCOME_TYPE,NAME_FAMILY_STATUS,OCCUPATION_TYPE,AMT_CREDIT,age
0,1,1,0.142129,4,2,Working,Single / not married,Laborers,406597.5,-1.542178
1,0,0,0.426790,1,1,State servant,Married,Core staff,1293502.5,0.130824
2,0,1,-0.427192,4,2,Working,Single / not married,Laborers,135000.0,0.716375
